# Basic Map Visualization with Lonboard

This notebook demonstrates using lonboard for geospatial visualization and how to connect custom widgets to maps.

In [1]:
import sys
import pathlib
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import lonboard
from lonboard import Map, ScatterplotLayer
import ipywidgets as widgets
from IPython.display import display

# Add parent directory to import our widgets
sys.path.insert(0, str(pathlib.Path().absolute().parent))

## Generate Sample Geospatial Data

Let's create some sample point data to visualize on the map:

In [2]:
# Generate random points around a few city centers
np.random.seed(42)

cities = {
    'San Francisco': (-122.4194, 37.7749),
    'New York': (-74.0060, 40.7128),
    'Chicago': (-87.6298, 41.8781),
    'Los Angeles': (-118.2437, 34.0522),
    'Seattle': (-122.3321, 47.6062)
}

# Generate points around each city
points_data = []
for city, (lon, lat) in cities.items():
    n_points = np.random.randint(50, 150)
    for i in range(n_points):
        # Add some random offset
        point_lon = lon + np.random.normal(0, 0.1)
        point_lat = lat + np.random.normal(0, 0.1)
        value = np.random.randint(1, 100)
        category = np.random.choice(['A', 'B', 'C'])
        
        points_data.append({
            'city': city,
            'geometry': Point(point_lon, point_lat),
            'value': value,
            'category': category
        })

# Create GeoDataFrame
gdf = gpd.GeoDataFrame(points_data, crs='EPSG:4326')
print(f"Created {len(gdf)} points around {len(cities)} cities")
gdf.head()

Created 530 points around 5 cities


,city,geometry,value,category
0,San Francisco,POINT (-122.53059 37.80679),83,C
1,San Francisco,POINT (-122.26148 37.85164),22,A
2,San Francisco,POINT (-122.47749 37.72238),21,A
3,San Francisco,POINT (-122.3952 37.58357),89,A
4,San Francisco,POINT (-122.66331 37.83524),92,C


## Basic Lonboard Map

In [3]:
# Create a basic lonboard map
layer = ScatterplotLayer.from_geopandas(
    gdf,
    get_radius=10000,  # 10km radius
    get_fill_color=[255, 0, 0, 100],  # Red with transparency
    pickable=True
)

m = Map(
    layers=[layer],
    view_state={
        'longitude': -98.5795,
        'latitude': 39.8283,
        'zoom': 3,
        'pitch': 0,
        'bearing': 0
    },
    height=400
)

m

## Color by Category

In [4]:
# Define colors for each category
color_map = {
    'A': [255, 0, 0, 180],    # Red
    'B': [0, 255, 0, 180],    # Green
    'C': [0, 0, 255, 180]     # Blue
}

# Map categories to colors — lonboard needs a numpy array, not a list of lists
colors = np.array(gdf['category'].map(color_map).tolist(), dtype=np.uint8)

# Create colored layer
colored_layer = ScatterplotLayer.from_geopandas(
    gdf,
    get_radius=10000,
    get_fill_color=colors,
    pickable=True
)

m2 = Map(
    layers=[colored_layer],
    view_state={
        'longitude': -98.5795,
        'latitude': 39.8283,
        'zoom': 3,
        'pitch': 0,
        'bearing': 0
    },
    height=400
)

m2

## Size by Value

In [5]:
# Scale radius by value
# Normalize values to get reasonable radius sizes
min_radius = 5000
max_radius = 30000
normalized_values = (gdf['value'] - gdf['value'].min()) / (gdf['value'].max() - gdf['value'].min())
radii = np.asarray(min_radius + normalized_values * (max_radius - min_radius), dtype=np.float32)

# Create sized layer
sized_layer = ScatterplotLayer.from_geopandas(
    gdf,
    get_radius=radii,
    get_fill_color=colors,
    pickable=True,
    auto_highlight=True
)

m3 = Map(
    layers=[sized_layer],
    view_state={
        'longitude': -98.5795,
        'latitude': 39.8283,
        'zoom': 3,
        'pitch': 30,
        'bearing': 0
    },
    height=400
)

m3

## Interactive Filtering with Widgets

Now let's create custom widgets to filter the map data:

In [7]:
# Create filtering widgets
category_selector = widgets.SelectMultiple(
    options=['A', 'B', 'C'],
    value=['A', 'B', 'C'],
    description='Categories:',
    disabled=False
)

value_slider = widgets.IntRangeSlider(
    value=[0, 100],
    min=0,
    max=100,
    step=1,
    description='Value Range:',
    continuous_update=False,
    readout=True
)

city_selector = widgets.SelectMultiple(
    options=list(cities.keys()),
    value=list(cities.keys()),
    description='Cities:',
    disabled=False
)

# Function to update map based on filters
def update_map(*args):
    # Filter data
    filtered_gdf = gdf[
        (gdf['category'].isin(category_selector.value)) &
        (gdf['value'] >= value_slider.value[0]) &
        (gdf['value'] <= value_slider.value[1]) &
        (gdf['city'].isin(city_selector.value))
    ]
    
    if len(filtered_gdf) == 0:
        # If no data, clear the map
        m4.layers = []
        stats_output.value = "No data matches the current filters"
        return
    
    # Update colors and sizes
    fcolors = np.array(filtered_gdf['category'].map(color_map).tolist(), dtype=np.uint8)
    normalized_values = (filtered_gdf['value'] - filtered_gdf['value'].min()) / \
                       (filtered_gdf['value'].max() - filtered_gdf['value'].min() + 0.001)
    fradii = np.asarray(min_radius + normalized_values * (max_radius - min_radius), dtype=np.float32)
    
    # Create new layer
    new_layer = ScatterplotLayer.from_geopandas(
        filtered_gdf,
        get_radius=fradii,
        get_fill_color=fcolors,
        pickable=True,
        auto_highlight=True
    )
    
    # Update map
    m4.layers = [new_layer]
    
    # Update statistics
    stats_output.value = f"""
    <b>Filtered Statistics:</b><br>
    Total Points: {len(filtered_gdf)}<br>
    Average Value: {filtered_gdf['value'].mean():.1f}<br>
    Categories: {', '.join(filtered_gdf['category'].value_counts().to_dict().keys())}
    """

# Connect widgets to update function
category_selector.observe(update_map, 'value')
value_slider.observe(update_map, 'value')
city_selector.observe(update_map, 'value')

# Statistics display
stats_output = widgets.HTML(
    value="""
    <b>Statistics:</b><br>
    Total Points: {}<br>
    Average Value: {:.1f}
    """.format(len(gdf), gdf['value'].mean())
)

# Create the interactive map
m4 = Map(
    layers=[sized_layer],
    view_state={
        'longitude': -98.5795,
        'latitude': 39.8283,
        'zoom': 3,
        'pitch': 0,
        'bearing': 0
    },
    height=500
)

# Layout
controls = widgets.VBox([
    widgets.HTML("<h3>Filter Controls</h3>"),
    category_selector,
    value_slider,
    city_selector,
    stats_output
])

widgets.HBox([controls, m4])

## Focus on Specific City

Let's create a widget to zoom to specific cities:

In [8]:
# Create city focus widget
city_dropdown = widgets.Dropdown(
    options=list(cities.keys()),
    value='San Francisco',
    description='Focus City:'
)

zoom_slider = widgets.IntSlider(
    value=10,
    min=3,
    max=15,
    description='Zoom Level:'
)

def focus_city(change):
    city = city_dropdown.value
    lon, lat = cities[city]
    
    # Filter data for this city
    city_gdf = gdf[gdf['city'] == city]
    
    # Create layer for this city
    ccolors = np.array(city_gdf['category'].map(color_map).tolist(), dtype=np.uint8)
    city_layer = ScatterplotLayer.from_geopandas(
        city_gdf,
        get_radius=1000,  # Smaller radius for city view
        get_fill_color=ccolors,
        pickable=True
    )
    
    # Update map
    m5.layers = [city_layer]
    m5.view_state = {
        'longitude': lon,
        'latitude': lat,
        'zoom': zoom_slider.value,
        'pitch': 0,
        'bearing': 0
    }
    
    city_stats.value = f"""
    <b>{city} Statistics:</b><br>
    Points: {len(city_gdf)}<br>
    Avg Value: {city_gdf['value'].mean():.1f}<br>
    Categories: A={len(city_gdf[city_gdf['category']=='A'])}, 
                B={len(city_gdf[city_gdf['category']=='B'])}, 
                C={len(city_gdf[city_gdf['category']=='C'])}
    """

city_dropdown.observe(focus_city, 'value')
zoom_slider.observe(focus_city, 'value')

city_stats = widgets.HTML(value="")

# Create map focused on first city
m5 = Map(
    layers=[],
    view_state={
        'longitude': cities['San Francisco'][0],
        'latitude': cities['San Francisco'][1],
        'zoom': 10,
        'pitch': 0,
        'bearing': 0
    },
    height=400
)

# Initial focus
focus_city(None)

# Layout
city_controls = widgets.VBox([
    widgets.HTML("<h3>City Focus</h3>"),
    city_dropdown,
    zoom_slider,
    city_stats
])

widgets.HBox([city_controls, m5])

## Summary

In this notebook we demonstrated:

1. **Basic lonboard maps** - Creating simple point visualizations
2. **Styling by attributes** - Coloring by category, sizing by value
3. **Interactive filtering** - Using widgets to filter map data
4. **Dynamic view updates** - Focusing on specific regions
5. **Widget-map integration** - Connecting standard widgets with lonboard

### Key Points

- Lonboard provides fast, GPU-accelerated rendering
- Works seamlessly with GeoPandas data
- Can handle large datasets efficiently
- Integrates well with ipywidgets for interactivity

Next, we'll explore more complex inter-widget communication patterns with maps.